# LLM Audit - Demonstração Completa

Este notebook demonstra como usar o módulo `llm_audit` para:
1. Auditar imóveis suspeitos via LLM
2. Extrair features qualitativas
3. Fazer merge dos resultados na base de ML

## Setup

Certifique-se de ter as dependências instaladas e a chave da API OpenAI configurada.

In [ ]:
import os
import json
from pathlib import Path
import pandas as pd

# Verificar OPENAI_API_KEY
if not os.environ.get('OPENAI_API_KEY'):
    raise ValueError('OPENAI_API_KEY não está configurada. Execute: os.environ["OPENAI_API_KEY"] = "sua-chave"')

print("✓ OPENAI_API_KEY detectada")

In [ ]:
# Imports do módulo llm_audit
from data_pipeline.llm_audit import run_audit
from data_pipeline.llm_audit.merge_audit_results import full_pipeline

# Setup de paths
PROJECT_ROOT = Path().cwd().parent if Path('data').exists() else Path().cwd()
SILVER_FILE = PROJECT_ROOT / 'data' / 'silver' / 'imoveis_limpos.parquet'

print(f"Project root: {PROJECT_ROOT}")
print(f"Silver file: {SILVER_FILE}")
print(f"Silver exists: {SILVER_FILE.exists()}")

## Etapa 1: Executar Auditoria LLM

Processa os imóveis e extrai veredictos + features qualitativas.

In [ ]:
# Executar auditoria (exemplo com limite para testes)
resultado = run_audit(
    project_root=PROJECT_ROOT,
    limit=5,  # Para teste - remover em produção
    only_suspicious=False,  # Auditar todos
    model_name="gpt-4o-mini",
    rate_limit_delay=1.0,
)

print(f"\n{json.dumps(resultado, indent=2)}")

In [ ]:
# Encontrar o arquivo de auditoria gerado
audit_dir = PROJECT_ROOT / 'data' / 'auditoria'
audit_files = sorted(audit_dir.glob('auditoria_llm_*.json'), reverse=True)

if audit_files:
    latest_audit = audit_files[0]
    print(f"✓ Último arquivo de auditoria: {latest_audit.name}")
else:
    print("✗ Nenhum arquivo de auditoria encontrado")

In [ ]:
# Inspecionar conteúdo da auditoria
with open(latest_audit, 'r', encoding='utf-8') as f:
    audit_data = json.load(f)

print(f"Metadados:")
print(json.dumps(audit_data['metadados'], indent=2))

print(f"\nPrimeiro resultado de auditoria:")
if audit_data['auditoria']:
    print(json.dumps(audit_data['auditoria'][0], indent=2))

## Etapa 2: Fazer Merge com a Base de ML

Integra os resultados de auditoria (veredictos + features) na base Silver/Gold.

In [ ]:
# Opção 1: Apenas merge sem aplicar veredictos (para inspeção)
from data_pipeline.llm_audit.merge_audit_results import merge_audit_with_silver

merged_df = merge_audit_with_silver(
    silver_parquet=SILVER_FILE,
    audit_json=latest_audit,
    output_parquet=None,  # Não salvar por enquanto
)

print(f"\nDataFrame merged shape: {merged_df.shape}")
print(f"\nColunas adicionadas:")
print(merged_df.filter(like='llm_').columns.tolist())
print(merged_df.filter(like='veredicto').columns.tolist())

In [ ]:
# Inspecionar alguns resultados
audit_cols = ['id_hex', 'titulo', 'veredicto', 'justificativa', 
               'llm_mobiliado', 'llm_reformado', 'llm_tem_lazer', 'llm_ar_condicionado']

# Filtrar apenas registros com auditoria
com_auditoria = merged_df[merged_df['veredicto'].notna()]

if len(com_auditoria) > 0:
    print(f"Registros com auditoria: {len(com_auditoria)}")
    print(f"\nExemplos:")
    print(com_auditoria[audit_cols].head(3).to_string(index=False))
else:
    print("Nenhum registro com auditoria encontrado")

In [ ]:
# Opção 2: Pipeline completo (merge + aplicação de veredictos)
output_enriched = PROJECT_ROOT / 'data' / 'silver' / 'imoveis_com_auditoria.parquet'

resultado_merge = full_pipeline(
    silver_parquet=SILVER_FILE,
    audit_json=latest_audit,
    output_parquet=output_enriched,
)

print(f"\nResultado do merge:")
print(json.dumps(resultado_merge, indent=2))

## Etapa 3: Análise dos Resultados

Estatísticas sobre as auditorias e features extraídas.

In [ ]:
# Carregar base enriquecida
enriquecida = pd.read_parquet(output_enriched)

print(f"Base enriquecida: {len(enriquecida)} registros")
print(f"\nDistribuição de veredictos:")
print(enriquecida['veredicto'].value_counts(dropna=False))

# Features qualitativas
print(f"\nFeatures qualitativas extraídas:")
for col in ['llm_mobiliado', 'llm_reformado', 'llm_tem_lazer', 'llm_ar_condicionado']:
    if col in enriquecida.columns:
        count = enriquecida[col].sum()
        print(f"  - {col}: {count} imóveis ({count/len(enriquecida)*100:.1f}%)")

In [ ]:
# Comparar antes vs depois (se houve correções)
corrigidos = enriquecida[enriquecida['veredicto'] == 'CORRIGIR']

if len(corrigidos) > 0:
    print(f"Imóveis corrigidos: {len(corrigidos)}")
    print(f"\nExemplo de correção:")
    cols = ['id_hex', 'area_m2', 'area_m2_corrigida', 'vagas', 'vagas_corrigidas', 'justificativa']
    print(corrigidos[cols].head(1).to_string(index=False))
else:
    print("Nenhum imóvel foi corrigido")

## Etapa 4: Próximos Passos

Como usar as features extraídas no treinamento do modelo.

In [ ]:
# A base enriquecida agora pode ser usada no pipeline de treinamento
# Exemplo: Usar features qualitativas como input para o modelo

from data_pipeline.medallion.silver_to_gold import load_silver, engineer_features

# Carregar a base enriquecida
base_com_features = pd.read_parquet(output_enriched)

# Filtrar apenas registros não excluídos
base_valida = base_com_features[base_com_features['veredicto'] != 'EXCLUIR'].copy()

print(f"Base válida para treinamento: {len(base_valida)} registros")

# As features qualitativas estão disponíveis para uso:
feature_cols = ['llm_mobiliado', 'llm_reformado', 'llm_tem_lazer', 'llm_ar_condicionado']
print(f"\nFeatures disponíveis para treinamento: {feature_cols}")

## Resumo

O módulo `llm_audit` fornece:

1. **Auditoria de dados suspeitos**: Veredicto MANTER/CORRIGIR/EXCLUIR
2. **Extração de features qualitativas**: mobiliado, reformado, lazer, AR
3. **Integração segura**: Append-only JSON, merge via pandas
4. **Rastreabilidade completa**: Metadados, timestamp, modelo LLM

Output final enriquecido em `imoveis_com_auditoria.parquet`.